# Chapter 8 — Computer Vision for Automated Student Behaviour Assessment

**Companion notebook · annotated classroom images · YOLO object detection · observational analytics**

This notebook organises dataset inspection, class distributions, annotated-image examples, optional detector training, held-out evaluation, and observational spatial–temporal summaries. Charts are calculated from the supplied annotations or actual detector outputs; no performance values are hard-coded.

**Source:** [studentactivity, *scb-dataset-3*, version 3 (Roboflow Universe, 2024)](https://universe.roboflow.com/studentactivity/scb-dataset-3/dataset/3), licensed under **CC BY 4.0**. The published v3 export has **5,019 images** (3,520 train, 996 validation, 503 test), resized to 256 × 256 without dataset-level augmentation.

**Important source distinction:** The specified Roboflow project describes the labels **reading**, **hand-raising**, and **writing**. Chapter 8 discusses *Doing Homework*, *Listening*, and *Looking Around*. These are **not equivalent categories**. The notebook reads the actual classes from `data.yaml`; it never silently renames them. Results produced from this dataset are companion analyses and **do not reproduce** the chapter’s historical three-class figures or YOLOv7 confusion matrix without the original corresponding dataset, weights, and evaluation artefacts.

**Setup:** In the notebook environment install `ultralytics`, `numpy`, `pandas`, `matplotlib`, `Pillow`, `PyYAML`, and `opencv-python` as needed. Export version 3 of the source dataset in **YOLOv8** format, then unpack it under `data/SCB-Dataset3-v3/`, retaining its `data.yaml`, `train/`, `valid/`, and `test/` folders. Keep the dataset outside the public repository unless its redistribution terms are explicitly reviewed.

Training, inference, and video analyses are opt-in. No input image is sent to a remote detection service by this notebook.

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>01 · Imports and reproducible settings</strong><div style="margin-top:4px;font-size:0.96em;">Define input locations, output folders, and explicit switches for computationally expensive stages.</div></div>

In [ ]:
from pathlib import Path
from collections import Counter
import json, math, os, random, shutil, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw, ImageOps

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path('data/SCB-Dataset3-v3')
OUT = Path('outputs/chapter08')
FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

RUN_TRAINING = False         # Change only when training is intended.
RUN_TEST_EVALUATION = True  # Requires trained detector weights.
RUN_VIDEO_ANALYSIS = False  # Requires a genuine time-ordered recording.
VIDEO_PATH = Path('data/classroom_session.mp4')
CHECKPOINT = Path('outputs/chapter08/training/yolov8n_v3/weights/best.pt')
MODEL_TEMPLATE = 'yolov8n.pt'  # Accessible companion YOLO implementation, not historical YOLOv7.
IMAGE_SIZE, EPOCHS, BATCH_SIZE = 256, 50, 16
CONFIDENCE_FOR_INFERENCE = 0.25

try:
    import cv2
except ImportError:
    cv2 = None

print('Working directory:', Path.cwd())
print('Dataset directory:', DATA_ROOT)
print('Training enabled:', RUN_TRAINING)

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>02 · Locate the original annotation export</strong><div style="margin-top:4px;font-size:0.96em;">Validate the class names and split directories without altering the annotations or the supplied partition.</div></div>

In [ ]:
possible_yamls = [DATA_ROOT / 'data.yaml', DATA_ROOT / 'dataset.yaml']
source_yaml = next((p for p in possible_yamls if p.is_file()), None)
if source_yaml is None and DATA_ROOT.exists():
    candidates = sorted(DATA_ROOT.glob('*/data.yaml'))
    source_yaml = candidates[0] if len(candidates) == 1 else None

DATASET_OK = source_yaml is not None
class_names = []
split_dirs = {}
runtime_yaml = None

if not DATASET_OK:
    print('Dataset not installed. Export YOLOv8 files from the linked Roboflow v3 page,')
    print('then place data.yaml and train/valid/test directories in', DATA_ROOT)
else:
    root = source_yaml.parent.resolve()
    config = yaml.safe_load(source_yaml.read_text(encoding='utf-8'))
    names = config.get('names', [])
    if isinstance(names, dict):
        class_names = [str(names[i] if i in names else names[str(i)]) for i in range(len(names))]
    else:
        class_names = [str(x) for x in names]
    if len(class_names) == 0:
        raise ValueError('No category names are declared in data.yaml')
    for key, aliases in {'train': ['train'], 'val': ['valid', 'val'], 'test': ['test']}.items():
        split_dirs[key] = next((root / alias for alias in aliases if (root / alias / 'images').is_dir()), None)
    if any(v is None for v in split_dirs.values()):
        raise FileNotFoundError('Expected train, valid/val and test folders with images/ subfolders')

    # The original partitions are preserved, but their local paths are made portable.
    portable = {'path': str(root),
                'train': str((split_dirs['train'] / 'images').resolve()),
                'val': str((split_dirs['val'] / 'images').resolve()),
                'test': str((split_dirs['test'] / 'images').resolve()),
                'nc': len(class_names), 'names': dict(enumerate(class_names))}
    runtime_yaml = (OUT / 'dataset_runtime.yaml').resolve()
    runtime_yaml.write_text(yaml.safe_dump(portable, sort_keys=False), encoding='utf-8')
    print('Source YAML:', source_yaml)
    print('Dataset labels:', class_names)
    print('Declared classes:', len(class_names))
    chapter_labels = {'doing homework', 'listening', 'looking around'}
    source_labels = {name.lower().replace('_', ' ').replace('-', ' ') for name in class_names}
    if source_labels != chapter_labels:
        print('SOURCE MISMATCH: Book labels and these Roboflow categories are different.')
        print('Do not use this export to claim replication of chapter-specific results.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>03 · Annotation inspection and dataset audit</strong><div style="margin-top:4px;font-size:0.96em;">Read YOLO bounding-box labels, report class frequencies and identify malformed annotation rows.</div></div>

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
rows, image_rows, issues = [], [], []

if DATASET_OK:
    for split, folder in split_dirs.items():
        image_folder = folder / 'images'
        for image_path in sorted(p for p in image_folder.rglob('*') if p.suffix.lower() in EXTENSIONS):
            rel = image_path.relative_to(image_folder)
            label_path = folder / 'labels' / rel.with_suffix('.txt')
            image_rows.append({'split': split, 'image': str(image_path), 'label': str(label_path)})
            if not label_path.is_file():
                issues.append(f'Missing annotation file: {label_path}')
                continue
            for lineno, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
                parts = line.split()
                if len(parts) != 5:
                    issues.append(f'Invalid YOLO row in {label_path}:{lineno}')
                    continue
                try:
                    vals = [float(v) for v in parts]
                    cid, x, y, w, h = vals
                    valid = (cid.is_integer() and 0 <= int(cid) < len(class_names)
                             and 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1)
                    if not valid:
                        raise ValueError('Value outside the expected range')
                    rows.append({'split': split, 'image': str(image_path), 'class_id': int(cid),
                                 'class_name': class_names[int(cid)], 'x': x, 'y': y, 'w': w, 'h': h})
                except ValueError:
                    issues.append(f'Invalid values in {label_path}:{lineno}')

ann = pd.DataFrame(rows, columns=['split', 'image', 'class_id', 'class_name', 'x', 'y', 'w', 'h'])
image_index = pd.DataFrame(image_rows, columns=['split','image','label'])
if DATASET_OK:
    print('Images per split:', image_index.groupby('split').size().to_dict())
    print('Annotated instances:', len(ann))
    print('Annotation issues:', len(issues))
    if issues:
        print('Example issue:', issues[0])
    if len(ann):
        print(ann.groupby(['split', 'class_name']).size().unstack(fill_value=0).to_string())
        ann.groupby(['split', 'class_name']).size().rename('instances').reset_index().to_csv(OUT / 'annotation_counts.csv', index=False)
else:
    print('Annotation audit skipped: dataset not found.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>04 · Behaviour-category frequencies (Figure 8.2 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Calculate annotation counts from the actual export; do not substitute the historical values printed in the book.</div></div>

In [ ]:
def save_plot(fig, stem):
    fig.savefig(FIG / f'{stem}.png', dpi=220, bbox_inches='tight')
    fig.savefig(FIG / f'{stem}.pdf', bbox_inches='tight')
    plt.close(fig)

if len(ann):
    counts = ann['class_name'].value_counts().reindex(class_names, fill_value=0)
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    bars = ax.bar(counts.index, counts.values)
    ax.bar_label(bars, padding=3, fmt='%d')
    ax.set(ylabel='Annotated instances', title='Observed behaviour categories (Roboflow v3)')
    ax.tick_params(axis='x', labelrotation=18)
    fig.tight_layout()
    save_plot(fig, 'Figure_8_2_companion_class_distribution')
    print(counts.to_string())
else:
    print('Class distribution skipped: dataset labels are unavailable.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>05 · Class proportions (Figure 8.7 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Show each category’s observed proportion; this plot summarises annotations rather than partition-specific performance.</div></div>

In [ ]:
if len(ann):
    counts = ann['class_name'].value_counts().reindex(class_names, fill_value=0)
    positive = counts[counts > 0]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.pie(positive.values, labels=positive.index, autopct='%1.1f%%', startangle=110)
    ax.set_title('Annotated-instance proportions (Roboflow v3)')
    save_plot(fig, 'Figure_8_7_companion_category_proportions')
else:
    print('Category proportions skipped: annotations are unavailable.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>06 · Annotated classroom images (Figure 8.3 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Draw the supplied bounding boxes on sample training images to inspect visual crowding, overlap, and class labels.</div></div>

In [ ]:
def annotated_image(image_file, max_side=560):
    image_file = Path(image_file)
    im = Image.open(image_file).convert('RGB')
    width, height = im.size
    painter = ImageDraw.Draw(im)
    subset = ann[ann['image'] == str(image_file)]
    for obj in subset.itertuples():
        x0 = (obj.x - obj.w / 2) * width
        y0 = (obj.y - obj.h / 2) * height
        x1 = (obj.x + obj.w / 2) * width
        y1 = (obj.y + obj.h / 2) * height
        painter.rectangle((x0, y0, x1, y1), outline='#eab04a', width=max(2, width // 140))
        painter.text((max(x0, 0), max(y0 - 12, 0)), obj.class_name, fill='#eab04a', stroke_fill='black', stroke_width=1)
    im.thumbnail((max_side, max_side))
    return im

def montage(images, title, output_stem):
    if not images:
        print('No images available for', title)
        return
    ncols = 2
    nrows = int(math.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(9, 4.0 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, im in zip(axes, images):
        ax.imshow(im)
        ax.axis('off')
    for ax in axes[len(images):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=12)
    save_plot(fig, output_stem)

if DATASET_OK and len(image_index):
    samples = image_index.loc[image_index['split'] == 'train', 'image'].head(4).tolist()
    montage([annotated_image(p) for p in samples],
            'Example annotated training images (source labels)', 'Figure_8_3_companion_annotated_images')
else:
    print('Annotated-image preview skipped.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>07 · Training-batch illustration (Figure 8.4 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Inspect a reproducibly sampled group of labelled images before any model training.</div></div>

In [ ]:
if DATASET_OK and len(image_index):
    training_images = image_index.loc[image_index['split'] == 'train', 'image']
    chosen = training_images.sample(n=min(6, len(training_images)), random_state=SEED).tolist()
    montage([annotated_image(p) for p in chosen],
            'Sample labelled images (before training augmentation)',
            'Figure_8_4_companion_sample_batch')
else:
    print('Sample batch skipped.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>08 · Optional YOLO model training</strong><div style="margin-top:4px;font-size:0.96em;">Train a reproducible companion YOLOv8n detector using the original export partitions; this is not a reproduction of the chapter’s historical YOLOv7 run.</div></div>

In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

train_result = None
if RUN_TRAINING and DATASET_OK:
    if YOLO is None:
        raise ImportError('Install ultralytics before enabling RUN_TRAINING.')
    detector = YOLO(MODEL_TEMPLATE)
    train_result = detector.train(
        data=str(runtime_yaml), imgsz=IMAGE_SIZE, epochs=EPOCHS, batch=BATCH_SIZE,
        seed=SEED, deterministic=True, patience=12,
        fliplr=0.5, degrees=5.0, translate=0.1, scale=0.2, hsv_v=0.3,
        project=str((OUT / 'training').resolve()), name='yolov8n_v3', exist_ok=True)
    print('Training output:', detector.trainer.save_dir)
else:
    print('Training disabled. No model performance is claimed or generated.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>09 · Training dynamics from recorded losses</strong><div style="margin-top:4px;font-size:0.96em;">Plot training and validation loss only when an actual training log exists.</div></div>

In [ ]:
results_csv = OUT / 'training' / 'yolov8n_v3' / 'results.csv'
if results_csv.is_file():
    history = pd.read_csv(results_csv)
    history.columns = history.columns.str.strip()
    keys = [k for k in history.columns if k.startswith(('train/','val/')) and k.endswith('_loss')]
    if keys:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for k in keys:
            ax.plot(history['epoch'] + 1, history[k], label=k, linewidth=1.4)
        ax.set(xlabel='Epoch', ylabel='Recorded loss', title='Detector training and validation losses')
        ax.legend(fontsize=8)
        save_plot(fig, 'Figure_8_4_companion_training_dynamics')
else:
    print('No training log found; loss curves were not fabricated.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>10 · Independent test-set evaluation</strong><div style="margin-top:4px;font-size:0.96em;">Evaluate an existing checkpoint on the untouched test partition and retain the library-generated detection diagnostics.</div></div>

In [ ]:
results_test = None
model = None
checkpoint_ready = CHECKPOINT.is_file()
if DATASET_OK and checkpoint_ready and RUN_TEST_EVALUATION:
    if YOLO is None:
        raise ImportError('Install ultralytics before evaluating the saved model.')
    model = YOLO(str(CHECKPOINT))
    results_test = model.val(data=str(runtime_yaml), split='test', imgsz=IMAGE_SIZE,
                             conf=0.001, iou=0.7, plots=True, seed=SEED,
                             project=str((OUT / 'evaluation').resolve()), name='test', exist_ok=True)
    box = results_test.box
    report = {'precision': float(box.mp), 'recall': float(box.mr),
              'mAP50': float(box.map50), 'mAP50_95': float(box.map),
              'test_images': int((image_index['split'] == 'test').sum()),
              'model_checkpoint': str(CHECKPOINT)}
    (OUT / 'test_metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(pd.Series(report).to_string())
else:
    print('Test evaluation skipped: requires dataset and a trained checkpoint at', CHECKPOINT)
    print('Training and evaluation must be run before quality metrics are available.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>11 · Evaluation figures (Figures 8.6 and 8.8–8.11 companions)</strong><div style="margin-top:4px;font-size:0.96em;">Copy confusion matrices and confidence/precision–recall diagnostics produced by the actual evaluator, without retracing image pixels or assigning artificial values.</div></div>

In [ ]:
source_plots = {
    'confusion_matrix_normalized.png': 'Figure_8_6_companion_normalised_confusion',
    'PR_curve.png': 'Figure_8_8_companion_precision_recall',
    'F1_curve.png': 'Figure_8_9_companion_F1_confidence',
    'P_curve.png': 'Figure_8_10_companion_precision_confidence',
    'R_curve.png': 'Figure_8_11_companion_recall_confidence',
}
if results_test is not None:
    eval_folder = Path(results_test.save_dir)
    for src, stem in source_plots.items():
        path = eval_folder / src
        if path.exists():
            shutil.copyfile(path, FIG / f'{stem}.png')
            # The PDF embeds the source raster graphic; the metrics are not re-estimated.
            Image.open(path).convert('RGB').save(FIG / f'{stem}.pdf', 'PDF', resolution=220)
            print('Exported:', stem)
        else:
            print('Unavailable from this evaluator:', src)
else:
    print('Evaluation graphs skipped: no evaluated test predictions.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>12 · Summary of detection metrics (Figure 8.12 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Visualise observed precision, recall, mAP@50, and mAP@50–95 from the held-out test evaluation.</div></div>

In [ ]:
if results_test is not None:
    stats = {'Precision': results_test.box.mp, 'Recall': results_test.box.mr,
             'mAP@50': results_test.box.map50, 'mAP@50–95': results_test.box.map}
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    bars = ax.bar(stats.keys(), stats.values())
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set(ylim=(0, 1.08), ylabel='Score', title='Held-out test metrics (companion experiment)')
    save_plot(fig, 'Figure_8_12_companion_test_summary')
else:
    print('Metric summary skipped: no test evaluation.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>13 · Representative test predictions (Figure 8.5 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Visualise bounding boxes predicted by the trained checkpoint on held-out images.</div></div>

In [ ]:
if model is not None and DATASET_OK and cv2 is not None:
    test_files = image_index.loc[image_index['split'] == 'test', 'image'].head(4).tolist()
    predicted = model.predict(source=test_files, imgsz=IMAGE_SIZE,
                              conf=CONFIDENCE_FOR_INFERENCE, verbose=False)
    previews = [Image.fromarray(cv2.cvtColor(item.plot(), cv2.COLOR_BGR2RGB)) for item in predicted]
    montage(previews, 'Held-out test predictions (companion detector)',
            'Figure_8_5_companion_test_predictions')
else:
    print('Test prediction montage skipped: checkpoint and dataset are required.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>14 · Spatial annotation density (Figure 8.13 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Aggregate bounding-box centroids for a category in normalised image coordinates; avoid interpreting image position as a validated seating location.</div></div>

In [ ]:
if len(ann):
    preferred = next((name for name in class_names if name.lower() == 'looking around'), None)
    spatial_category = preferred or class_names[0]
    subset = ann.loc[(ann['split'] == 'test') & (ann['class_name'] == spatial_category)]
    if len(subset):
        density, xedges, yedges = np.histogram2d(subset['y'], subset['x'], bins=(20,20), range=((0,1),(0,1)))
        fig, ax = plt.subplots(figsize=(6,5))
        image = ax.imshow(density, extent=(0,1,1,0), cmap='Blues', interpolation='nearest')
        ax.set(xlabel='Normalised image x', ylabel='Normalised image y',
               title=f'Annotation-centroid density: {spatial_category}')
        fig.colorbar(image, ax=ax, label='Annotation count')
        save_plot(fig, 'Figure_8_13_companion_spatial_annotation_density')
        print('This plot does not establish attention, seating layout, or classroom-zone effects.')
    else:
        print('No test annotations for:', spatial_category)
else:
    print('Spatial density skipped: annotations unavailable.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>15 · Spatial comparison by category (Figure 8.16 companion)</strong><div style="margin-top:4px;font-size:0.96em;">Compare spatial annotation distributions by observed category using the same normalised image coordinate system.</div></div>

In [ ]:
if len(ann):
    fig, axes = plt.subplots(1, len(class_names), figsize=(4.1 * len(class_names), 3.8), squeeze=False)
    for cid, label in enumerate(class_names):
        ax = axes[0, cid]
        frame = ann.loc[(ann['split'] == 'test') & (ann['class_name'] == label)]
        if len(frame):
            density, _, _ = np.histogram2d(frame['y'], frame['x'], bins=(20,20), range=((0,1),(0,1)))
            ax.imshow(density, extent=(0,1,1,0), cmap='Blues', interpolation='nearest')
        ax.set(title=f'{label}\n(n={len(frame)})', xlabel='Image x', ylabel='Image y')
    fig.suptitle('Spatial annotation patterns by category', fontsize=12)
    fig.tight_layout()
    save_plot(fig, 'Figure_8_16_companion_spatial_profiles')
else:
    print('Spatial category comparison skipped.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>16 · Optional time-ordered video analysis (Figures 8.14–8.15 companions)</strong><div style="margin-top:4px;font-size:0.96em;">Calculate a behavioural-detection proportion only when a genuine recording and an explicit, justified category mapping are available.</div></div>

In [ ]:
# Supply pedagogically reviewed category assignments before activating this analysis.
ON_TASK_CLASSES = []
OFF_TASK_CLASSES = []
PROCESS_EVERY_N_FRAMES = 30

can_process_video = (RUN_VIDEO_ANALYSIS and model is not None and VIDEO_PATH.is_file()
                     and cv2 is not None and ON_TASK_CLASSES and OFF_TASK_CLASSES)
if can_process_video:
    defined = set(ON_TASK_CLASSES) | set(OFF_TASK_CLASSES)
    if not defined.issubset(set(class_names)) or (set(ON_TASK_CLASSES) & set(OFF_TASK_CLASSES)):
        raise ValueError('On-task and off-task class assignments must be valid and disjoint')
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        raise ValueError('A valid frame rate is required for temporal analysis')
    frame_number, time_rows = 0, []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_number % PROCESS_EVERY_N_FRAMES == 0:
            prediction = model.predict(frame, imgsz=IMAGE_SIZE, conf=CONFIDENCE_FOR_INFERENCE, verbose=False)[0]
            labels = [class_names[int(idx)] for idx in prediction.boxes.cls.cpu().numpy()]
            on = sum(label in ON_TASK_CLASSES for label in labels)
            off = sum(label in OFF_TASK_CLASSES for label in labels)
            time_rows.append({'minute': frame_number / fps / 60, 'on': on, 'off': off,
                              'observed_proportion': on / (on + off) if (on + off) else np.nan})
        frame_number += 1
    cap.release()
    times = pd.DataFrame(time_rows).dropna(subset=['observed_proportion'])
    if len(times):
        times.to_csv(OUT / 'video_observations.csv', index=False)
        fig, ax = plt.subplots(figsize=(8,4))
        ax.plot(times['minute'], times['observed_proportion'], alpha=0.55, label='Frame-level proportion')
        ax.plot(times['minute'], times['observed_proportion'].rolling(5, min_periods=1).mean(),
                linewidth=2, label='5-sample rolling mean')
        ax.set(xlabel='Recording time (minutes)', ylabel='Proportion of mapped detections',
               ylim=(0,1), title='Observed class proportion over recorded time')
        ax.legend()
        save_plot(fig, 'Figure_8_14_companion_temporal_proportion')
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(times['minute'], times['on'], label='Selected on-task detections')
        ax.plot(times['minute'], times['off'], label='Selected off-task detections')
        ax.set(xlabel='Recording time (minutes)', ylabel='Detection count',
               title='Observed mapped categories over recorded time')
        ax.legend()
        save_plot(fig, 'Figure_8_15_companion_temporal_counts')
    else:
        print('Video contained no observations with the selected categories.')
else:
    print('Temporal analyses skipped. Static unordered photographs do not provide lesson-time observations.')
    print('To enable, provide a real video, trained weights, and reviewed class mappings.')
    print('Detection proportions are not direct measures of attention or learning.')

<div style="padding:10px 14px; border-left:4px solid #587d9b; border-radius:5px; background:#f1f5f8; margin:9px 0 5px 0;"><strong>17 · Output inventory and reproducibility record</strong><div style="margin-top:4px;font-size:0.96em;">Summarise the files produced and preserve the methodological settings needed to interpret them.</div></div>

In [ ]:
record = {
    'source': 'https://universe.roboflow.com/studentactivity/scb-dataset-3/dataset/3',
    'source_version': 3, 'source_license': 'CC BY 4.0',
    'dataset_available': DATASET_OK,
    'classes': class_names,
    'annotation_count': int(len(ann)),
    'image_count': int(len(image_index)),
    'annotation_issue_count': len(issues),
    'model_template': MODEL_TEMPLATE,
    'image_size': IMAGE_SIZE,
    'seed': SEED,
    'training_enabled': RUN_TRAINING,
    'test_evaluated': results_test is not None,
    'historical_book_results_reproduced': False,
    'note': 'The cited Roboflow v3 class names differ from the book categories. Companion analysis only.'
}
(OUT / 'notebook_run_manifest.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
exports = sorted(p.name for p in FIG.iterdir() if p.is_file())
print('Figures exported:', len(exports))
print('\n'.join(exports[:45]) if exports else '(none; dataset or trained model required)')
print('Saved:', OUT / 'notebook_run_manifest.json')

### Interpretation and replication boundaries

- The linked Roboflow version defines **reading**, **hand-raising**, and **writing**; these labels must not be presented as *Doing Homework*, *Listening*, and *Looking Around*.
- A YOLOv8n companion training run is **not** a replication of the chapter’s historical YOLOv7 results. Metrics and test plots appear only after a detector is actually trained and evaluated.
- Image-coordinate density does **not** establish classroom seating positions, distraction, or student psychological engagement. Static image partitions do not support lesson-time trends; those analyses require properly timestamped recordings and validated category definitions.
- Potential instructional use requires contextual interpretation, consent, privacy protections, and teacher review. Bounding-box predictions are observations of visible activity, not assessments of learning achievement.